In [1]:
import torch

from datasets import load_dataset

from transformers import (

    AutoTokenizer,

    AutoModelForCausalLM

)

from peft import LoraConfig

from trl import (

    DPOConfig,

    DPOTrainer

)

In [2]:
MODEL_NAME = "Qwen/Qwen2.5-0.5B"

tokenizer = AutoTokenizer.from_pretrained(

    MODEL_NAME,

    trust_remote_code=True

)

tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(

    MODEL_NAME,

    torch_dtype=torch.bfloat16,

    trust_remote_code=True

)

`torch_dtype` is deprecated! Use `dtype` instead!


In [4]:
dataset = load_dataset(

    "json",

    data_files="preference_dataset.json",

    split="train"

)

dataset

Generating train split: 0 examples [00:00, ? examples/s]

Dataset({
    features: ['prompt', 'chosen', 'rejected'],
    num_rows: 1
})

In [5]:
from peft import LoraConfig

peft_config = LoraConfig(

    r=16,

    lora_alpha=32,

    lora_dropout=0.05,

    bias="none",

    task_type="CAUSAL_LM",

    target_modules=[
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj"
    ]
)

In [6]:
from trl import DPOConfig

training_args = DPOConfig(

    output_dir="./dpo_outputs",

    num_train_epochs=3,

    per_device_train_batch_size=2,

    gradient_accumulation_steps=2,

    learning_rate=5e-6,

    logging_steps=5,

    save_strategy="epoch",

    report_to="none",

    bf16=True,

    remove_unused_columns=False
)

In [7]:
from trl import DPOTrainer

trainer = DPOTrainer(

    model=model,

    args=training_args,

    train_dataset=dataset,

    processing_class=tokenizer,

    peft_config=peft_config
)

Adding EOS to train dataset:   0%|          | 0/1 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/1 [00:00<?, ? examples/s]

[RANK 0] Mismatch between tokenized prompt and the start of tokenized prompt+chosen. This may be due to unexpected tokenizer behavior, whitespace issues, or special token handling. Verify that the tokenizer is processing text consistently.
[RANK 0] Mismatch between tokenized prompt and the start of tokenized prompt+rejected. This may be due to unexpected tokenizer behavior, whitespace issues, or special token handling. Verify that the tokenizer is processing text consistently.


In [8]:
trainer.train()

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151643}.


Step,Training Loss


TrainOutput(global_step=3, training_loss=0.6971009572347006, metrics={'train_runtime': 3.3918, 'train_samples_per_second': 0.884, 'train_steps_per_second': 0.884, 'total_flos': 2514664627200.0, 'train_loss': 0.6971009572347006, 'entropy': 0.7576308051745096, 'num_tokens': 1143.0, 'logits/chosen': -1.2643790122790215, 'logits/rejected': -1.0354567527770995, 'mean_token_accuracy': 0.7692307829856873, 'rewards/chosen': -0.017142995571096737, 'rewards/rejected': -0.009320704266428947, 'rewards/accuracies': 0.3333333333333333, 'rewards/margins': -0.007822291304667791, 'logps/chosen': -32.93426640828451, 'logps/rejected': -34.17059580485026, 'epoch': 3.0})

In [9]:
trainer.save_model("./qwen_dpo")

tokenizer.save_pretrained("./qwen_dpo")

('./qwen_dpo/tokenizer_config.json',
 './qwen_dpo/special_tokens_map.json',
 './qwen_dpo/chat_template.jinja',
 './qwen_dpo/vocab.json',
 './qwen_dpo/merges.txt',
 './qwen_dpo/added_tokens.json',
 './qwen_dpo/tokenizer.json')

## Test

In [13]:
from transformers import pipeline

pipe = pipeline(

    "text-generation",

    model="./qwen_dpo",

    tokenizer="./qwen_dpo"

)

prompt = """Clinical Question:

A junior orthopaedic surgery resident is completing a carpal tunnel repair with the department chairman as the attending physician. During the case, the resident inadvertently cuts a flexor tendon. The tendon is repaired without complication. The attending tells the resident that the patient will do fine, and there is no need to report this minor complication that will not harm the patient, as he does not want to make the patient worry unnecessarily. He tells the resident to leave this complication out of the operative report. Which of the following is the correct next action for the resident to take?

Options:
A. Disclose the error to the patient and put it in the operative report
B. Tell the attending that he cannot fail to disclose this mistake
C. Report the physician to the ethics committee
D. Refuse to dictate the operative report

Select the best answer.

Answer:
"""

result = pipe(

    prompt,

    max_new_tokens=128,

    do_sample=False

)

print(result[0]["generated_text"])

Device set to use cuda:0


Clinical Question:

A junior orthopaedic surgery resident is completing a carpal tunnel repair with the department chairman as the attending physician. During the case, the resident inadvertently cuts a flexor tendon. The tendon is repaired without complication. The attending tells the resident that the patient will do fine, and there is no need to report this minor complication that will not harm the patient, as he does not want to make the patient worry unnecessarily. He tells the resident to leave this complication out of the operative report. Which of the following is the correct next action for the resident to take?

Options:
A. Disclose the error to the patient and put it in the operative report
B. Tell the attending that he cannot fail to disclose this mistake
C. Report the physician to the ethics committee
D. Refuse to dictate the operative report

Select the best answer.

Answer:
A. Disclose the error to the patient and put it in the operative report

Explanation: The correct 

In [11]:
sample = dataset[0]

result = pipe(
    sample["prompt"],
    max_new_tokens=128,
    do_sample=False
)

print(result[0]["generated_text"])

Clinical Question:

A junior orthopaedic surgery resident is completing a carpal tunnel repair with the department chairman as the attending physician. During the case, the resident inadvertently cuts a flexor tendon. The tendon is repaired without complication. The attending tells the resident that the patient will do fine, and there is no need to report this minor complication that will not harm the patient, as he does not want to make the patient worry unnecessarily. He tells the resident to leave this complication out of the operative report. Which of the following is the correct next action for the resident to take?

Options:
A. Disclose the error to the patient and put it in the operative report
B. Tell the attending that he cannot fail to disclose this mistake
C. Report the physician to the ethics committee
D. Refuse to dictate the operative report

Select the best answer.


In [12]:
prompt = dataset[0]["prompt"]

result = pipe(
    prompt,
    max_new_tokens=128,
    do_sample=False,
    temperature=0.0
)

print(result[0]["generated_text"])

Clinical Question:

A junior orthopaedic surgery resident is completing a carpal tunnel repair with the department chairman as the attending physician. During the case, the resident inadvertently cuts a flexor tendon. The tendon is repaired without complication. The attending tells the resident that the patient will do fine, and there is no need to report this minor complication that will not harm the patient, as he does not want to make the patient worry unnecessarily. He tells the resident to leave this complication out of the operative report. Which of the following is the correct next action for the resident to take?

Options:
A. Disclose the error to the patient and put it in the operative report
B. Tell the attending that he cannot fail to disclose this mistake
C. Report the physician to the ethics committee
D. Refuse to dictate the operative report

Select the best answer.
